[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/tricycle/tricycle.ipynb)

# A Tricycle in the Plane and in Space

This example drives a front-steered tricycle across the ground in two and three dimensions. A wheel that rolls without sliding moves along its own heading, so at every moment it turns about a point on the line of its axle; all three wheels roll without sliding when all three axles pass through one turning centre, and the whole tricycle turns about that centre. Steering swings the turning centre from far away when the front wheel points nearly straight ahead to close beside the frame when it turns sharply.

Each axle is modelled as a plane of projective geometric algebra, which is a line in the plane and a vertical hyperplane in space ([Dorst and De Keninck, 2022](https://bivector.net/PGAdyn.pdf)). The turning centre is the meet `rear ^ front`, a point in the plane and a vertical line in space, and the tricycle steps in its body frame by the exponential `((rear ^ front) * (-step / 2)).exp()`, producing a rotation when the axles cross and a pure translation when they run parallel and meet at infinity. Steering turns the front axle by a motor `steer >> rest`, and accumulating body-frame step motors on the right traces the trajectory in both dimensions from the same module, following the [tricycle example of ganja.js](https://github.com/enkimute/ganja.js/blob/master/examples/example_chapter11_tricycle.html) by Steven De Keninck.

In the coordinate notation of the kinematic bicycle and Ackermann steering model on $\mathrm{SE}(2)$, front-wheel speed $v_f$ and steering angle $\delta$ integrate rear-axle midpoint coordinates $(x, y)$ and heading $\theta$ via $\dot x = v_f \cos\delta \cos\theta$, $\dot y = v_f \cos\delta \sin\theta$, and $\dot\theta = (v_f / L)\sin\delta$, while the instantaneous centre of rotation $(x - L \cot\delta \sin\theta,\, y + L \cot\delta \cos\theta)$ diverges as $\delta \to 0$ and must be replaced by straight-line translation in the zero-steering limit.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga.algebras import PGA2D, PGA3D
from examples import instantiate
from examples.util.animation import show_animation
from examples.mechanics.tricycle import render, scenarios

np.set_printoptions(precision=4, suppress=True)

## Turning centre where the axles meet

The rear axle runs across the origin of the tricycle's body frame, and the front axle at rest lies one wheelbase ahead, parallel to it. Steering turns the front axle about the steering axis where the front axle at rest meets the symmetry plane down the middle of the frame. Because the meet `rear ^ front` has weight equal to the sine of the steering angle, straight-ahead steering places the turning centre at infinity, while its distance from the front wheel equals the wheelbase divided by that sine.

In [ ]:
wheelbase = scenarios.wheelbase
steering = np.radians([0, 10, 30, 60])
n_angles = len(steering)

In [ ]:
plane = instantiate("examples.mechanics.tricycle.core", PGA2D)
mv = plane.mv
rear = mv.x                                                                     # [] Plane+
rest = mv.x - wheelbase * mv.w                                                  # [] Plane+
# Steering turns the front axle about its intersection with the symmetry plane y = 0.
axis = rest ^ mv.y                                                              # [] Meet+
steer = (axis.normalized() * (-steering / 2)).exp()                             # [n_angles] Motor+
front = steer >> rest                                                           # [n_angles] Plane+
centres = rear ^ front                                                          # [n_angles] Meet+
centre_points = centres * mv.antipseudoscalar([1])                              # [n_angles] Point-

In [ ]:
print("turning-centre weights vs sin(steering):", (mv.w & centre_points).to_array(), np.sin(steering))
print("turning-centre ground coordinates:\n", scenarios.ground_coordinates(centre_points))

## Drive in the plane

Across a drive the steering angle oscillates around a steady left turn. Each step moves the tricycle in its body frame by the exponential of that step's turning centre, so step motors multiply on the right of the pose.

In [ ]:
n_steps = scenarios.n_steps
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
plane_drive = scenarios.run(plane, n_steps)
show_animation(
    render.animate(render.plane, scenarios.moments(*plane_drive, n_frames)),
    "tricycle_plane",
    frame_duration_ms,
);

Red dashed extensions carry the rear and front axles out to the red turning-centre marker where they meet at every frame, while the grey curve traces the midpoint of the rear axle behind the dark frame and three rolling wheels.

## Drive in space

Instantiating the identical core module in three-dimensional projective geometric algebra turns each axle into a vertical plane and their meet into the vertical turning-centre line, leaving the same ground track up to floating-point round-off.

In [ ]:
space = instantiate("examples.mechanics.tricycle.core", PGA3D)
space_drive = scenarios.run(space, n_steps)
show_animation(
    render.animate(render.space, scenarios.moments(*space_drive, n_frames)),
    "tricycle_space",
    frame_duration_ms,
);

Red dashed axle rods meet at the vertical red post marking the turning-centre line, while the three wheel rims roll above the grey rear-axle trail across the ground.

In [ ]:
# checks
# Each turning centre's weight equals sin(steering) so straight-ahead axles meet at infinity, every wheel rolls
# along its heading up to chord curvature, and the planar and spatial drives leave the same ground track.
assert (((mv.w & centre_points) - np.sin(steering)).abs() < 1e-12).all()
for contacts, forward, _, _ in (plane_drive, space_drive):
    rolling = (contacts[:-1].normalized() & forward[:-1].normalized() & contacts[1:].normalized()).norm()  # [n_steps - 1, 3] Scalar+
    assert (rolling < 0.1 * wheelbase * (scenarios.step * wheelbase) ** 2).all()
assert (np.abs(scenarios.ground_coordinates(plane_drive[3]) - scenarios.ground_coordinates(space_drive[3])) < 1e-9).all()